# 文本特征提取 — BGE-large 三类特征

3 变体：
- **A: Desc only** — `title | description[:1500]`（基线，无 LLM）
- **B: LLM only** — `title | llm_v2`（纯 LLM ）
- **C: LLM+Desc** — `title | llm_v2 | description[:400]`（互补：LLM + description）

模型: `BAAI/bge-large-en-v1.5`

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch, pandas as pd, numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel

META_V2       = "final/item_meta_llm_enhanced_v2.csv"
META_RAW      = "final/item_meta_merged.csv"
OUTPUT_DIR    = "final/features"
MODEL_NAME    = "BAAI/bge-large-en-v1.5"
BATCH_SIZE    = 32   # RTX 5090 32GB

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  |  Batch: {BATCH_SIZE}")

Device: cuda  |  Batch: 32


### 加载 / 文本构造

In [ ]:
df_raw = pd.read_csv(META_RAW)
df_v2  = pd.read_csv(META_V2)

print(f"Raw meta:  {len(df_raw)} items")
print(f"LLM V2:    {len(df_v2)} items")
assert df_raw["parent_asin"].equals(df_v2["parent_asin"]), "ID order mismatch!"

item_ids  = df_v2["parent_asin"].tolist()
titles    = df_v2["title"].fillna("").tolist()
llm_texts = df_v2["llm_enhanced_text"].fillna("").tolist()
raw_descs = df_raw["description"].fillna("").tolist()

texts_a = [f"{t} | {d[:1500]}" for t, d in zip(titles, raw_descs)]
texts_b = [f"{t} | {l}" for t, l in zip(titles, llm_texts)]
texts_c = [f"{t} | {l} | {d[:400]}" for t, l, d in zip(titles, llm_texts, raw_descs)]

# token 预算检查
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
for label, texts in [("A: Desc only", texts_a), ("B: LLM only", texts_b), ("C: LLM+Desc", texts_c)]:
    tl = [len(tokenizer.encode(t, add_special_tokens=True)) for t in texts[:500]]
    over = sum(1 for x in tl if x > 512)
    print(f"{label:20s}: avg={np.mean(tl):.0f} tok  max={np.max(tl)}  >512: {over}")
del tokenizer

del df_raw, df_v2, titles, llm_texts, raw_descs
import gc; gc.collect()

Raw meta:  43528 items
LLM V2:    43528 items


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

A: Desc only        : avg=229 tok  max=406  >512: 0
B: LLM only         : avg=183 tok  max=241  >512: 0
C: LLM+Desc         : avg=263 tok  max=339  >512: 0


215

In [ ]:
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
model = model.to(dtype=torch.float16)

print(f"Model: {type(model).__name__}  |  dim: {model.config.hidden_size}  |  dtype: {model.dtype}")

Loading BAAI/bge-large-en-v1.5...


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model: BertModel  |  dim: 1024  |  dtype: torch.float16


###  Dataset + 提取函数

In [ ]:

class TextDataset(Dataset):
    def __init__(self, texts, item_ids):
        self.texts = texts
        self.ids = item_ids
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.ids[idx]

# BGE CLS-pooling: last_hidden_state[:, 0, :]"""
def extract_bge(model, tokenizer, dataloader, device, desc):
   
    model.eval()
    fts, ids = [], []
    with torch.no_grad():
        for texts, bids in tqdm(dataloader, desc=desc):
            inp = tokenizer(texts, return_tensors="pt",
                           padding=True, truncation=True, max_length=512).to(device)
            cls = model(**inp).last_hidden_state[:, 0, :].cpu().float().numpy()
            fts.append(cls)
            ids.extend(int(x) for x in bids)
    return np.concatenate(fts, axis=0), ids

### 变体 A — 仅 description

In [ ]:
print("A: Description only")
ds = TextDataset(texts_a, item_ids)
ldr = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
feat_a, ids_a = extract_bge(model, tokenizer, ldr, device, "BGE-A")
np.save(f"{OUTPUT_DIR}/text_bge_desc_1024.npy", feat_a)
pd.DataFrame({"item_id": ids_a}).to_csv(f"{OUTPUT_DIR}/text_id_map_bge_desc.csv", index=False)
print(f"  => {feat_a.shape}")
del texts_a, feat_a; gc.collect(); torch.cuda.empty_cache()

A: Description only


BGE-A: 100%|██████████| 1361/1361 [03:46<00:00,  6.02it/s]


  => (43528, 1024)


###  变体 B — 仅 LLM  

In [ ]:
print("B: LLM only")
ds = TextDataset(texts_b, item_ids)
ldr = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
feat_b, ids_b = extract_bge(model, tokenizer, ldr, device, "BGE-B")
np.save(f"{OUTPUT_DIR}/text_bge_llm_1024.npy", feat_b)
pd.DataFrame({"item_id": ids_b}).to_csv(f"{OUTPUT_DIR}/text_id_map_bge_llm.csv", index=False)
print(f"  => {feat_b.shape}")
del texts_b, feat_b; gc.collect(); torch.cuda.empty_cache()

B: LLM only


BGE-B: 100%|██████████| 1361/1361 [02:17<00:00,  9.91it/s]


  => (43528, 1024)


###  变体 C — LLM + Description

In [ ]:
print("C: LLM + Description")
ds = TextDataset(texts_c, item_ids)
ldr = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
feat_c, ids_c = extract_bge(model, tokenizer, ldr, device, "BGE-C")
np.save(f"{OUTPUT_DIR}/text_bge_llmdesc_1024.npy", feat_c)
pd.DataFrame({"item_id": ids_c}).to_csv(f"{OUTPUT_DIR}/text_id_map_bge_llmdesc.csv", index=False)
print(f"  => {feat_c.shape}")
del texts_c, feat_c; gc.collect(); torch.cuda.empty_cache()

C: LLM + Description


BGE-C: 100%|██████████| 1361/1361 [03:12<00:00,  7.08it/s]


  => (43528, 1024)


In [ ]:
print("=" * 50)
print("BGE-large 三变体完成")
print("=" * 50)

for label, path in [("A: Desc", "text_bge_desc_1024.npy"),
                    ("B: LLM",  "text_bge_llm_1024.npy"),
                    ("C: LLM+D","text_bge_llmdesc_1024.npy")]:
    full = f"{OUTPUT_DIR}/{path}"
    if os.path.exists(full):
        arr = np.load(full)
        print(f"  {label:10s} {str(arr.shape):15s} {arr.nbytes/1024**2:.0f} MB  NaN: {np.isnan(arr).any()}")

print("\nDone! Download final/features/ back to local.")

BGE-large 三变体完成
  A: Desc    (43528, 1024)   170 MB  NaN: False
  B: LLM     (43528, 1024)   170 MB  NaN: False
  C: LLM+D   (43528, 1024)   170 MB  NaN: False

Done! Download final/features/ back to local.
